# 21 — Clear-Sky Snapshot Planner

Replacement notebook with retry logic and caching for Open-Meteo.

In [ ]:

SITES = [
    {"site_id": "EST", "site_name": "Eastbourne",   "lat": 50.7680,  "lon": 0.2900},
    {"site_id": "NHV", "site_name": "Newhaven ERF", "lat": 50.80208, "lon": 0.04961},
    {"site_id": "LWS", "site_name": "Lewes",        "lat": 50.8739,  "lon": 0.0110},
]
DATE_FROM = "2024-12-26"
DATE_TO   = "2024-12-31"
SCENE_CATALOG_DIR = "outputs/s5p"
OUTPUT_DIR = "outputs/clear_sky_snapshot"
TOP_N_DAYS = 10


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json, math, hashlib, time
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars, retries=4, timeout=45):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": hourly_vars,
        "timezone": "UTC",
    }
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Open-Meteo request failed after {retries} attempts: {last_err}")

def weather_cache_path(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    safe = f"{lat}_{lon}_{start_date}_{end_date}_{hourly_vars}".replace(",", "_").replace("/", "_").replace(":", "_")
    return cache_dir / f"{safe}.json"

def fetch_weather_cached(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    p = weather_cache_path(cache_dir, lat, lon, start_date, end_date, hourly_vars)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    j = fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars)
    p.write_text(json.dumps(j), encoding="utf-8")
    return j

print("UTC now:", datetime.now(timezone.utc).isoformat())

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path(OUTPUT_DIR) / "weather_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

def fetch_weather(lat, lon, start_date, end_date):
    j = fetch_weather_cached(CACHE_DIR, lat, lon, start_date, end_date, "wind_speed_10m,cloud_cover,visibility")
    h = j.get("hourly", {})
    return pd.DataFrame({
        "time_utc": pd.to_datetime(h.get("time", []), utc=True, errors="coerce"),
        "wind_speed_ms": h.get("wind_speed_10m", []),
        "cloud_cover_pct": h.get("cloud_cover", []),
        "visibility_m": h.get("visibility", []),
    })
wx_rows = []
for site in SITES:
    w = fetch_weather(site["lat"], site["lon"], DATE_FROM, DATE_TO)
    w["site_id"] = site["site_id"]
    w["site_name"] = site["site_name"]
    w["date"] = w["time_utc"].dt.date.astype("string")
    wx_rows.append(w)
wx = pd.concat(wx_rows, ignore_index=True)


In [ ]:

scene_rows = []
for site in SITES:
    p = Path(SCENE_CATALOG_DIR) / f"{site['site_id']}_no2_offl_scene_catalog.csv"
    if p.exists():
        df = pd.read_csv(p)
        df["start_utc"] = pd.to_datetime(df["start_utc"], utc=True, errors="coerce")
        df["date"] = df["start_utc"].dt.date.astype("string")
        df["site_id"] = site["site_id"]
        df["site_name"] = site["site_name"]
        scene_rows.append(df)
scene = pd.concat(scene_rows, ignore_index=True) if scene_rows else pd.DataFrame()

daily = wx.groupby(["site_id","site_name","date"], dropna=False).agg(
    mean_cloud_cover_pct=("cloud_cover_pct","mean"),
    mean_wind_speed_ms=("wind_speed_ms","mean"),
    max_visibility_m=("visibility_m","max"),
).reset_index()
if not scene.empty:
    scene_daily = scene.groupby(["site_id","site_name","date"], dropna=False).agg(scene_count=("product_id","nunique")).reset_index()
    daily = daily.merge(scene_daily, on=["site_id","site_name","date"], how="left")
else:
    daily["scene_count"] = 0
daily["scene_count"] = daily["scene_count"].fillna(0)
daily["clarity_score"] = ((100 - daily["mean_cloud_cover_pct"].fillna(100)) / 100.0
                          + (daily["max_visibility_m"].fillna(0) / 10000.0)
                          - (daily["mean_wind_speed_ms"].fillna(99) / 10.0)
                          + (daily["scene_count"] > 0).astype(float))
shortlist = daily.sort_values(["clarity_score","scene_count"], ascending=[False, False]).groupby("site_id", dropna=False).head(TOP_N_DAYS).reset_index(drop=True)

daily_csv = Path(OUTPUT_DIR) / "clear_sky_daily_scores.csv"
short_csv = Path(OUTPUT_DIR) / "clear_sky_shortlist.csv"
daily.to_csv(daily_csv, index=False)
shortlist.to_csv(short_csv, index=False)
display(shortlist.head(20))


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))
for site in shortlist["site_id"].dropna().unique():
    s = shortlist[shortlist["site_id"] == site].copy()
    ax.scatter(pd.to_datetime(s["date"]), s["clarity_score"], label=site)
ax.set_title("Clear-sky snapshot shortlist")
ax.set_xlabel("Date")
ax.set_ylabel("Clarity score")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plot_path = Path(OUTPUT_DIR) / "clear_sky_shortlist_plot.png"
fig.savefig(plot_path, dpi=150)
plt.show()
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "clear_sky_snapshot",
    "sites": SITES,
    "date_from": DATE_FROM,
    "date_to": DATE_TO,
    "rows_weather": int(len(wx)),
    "rows_scene": int(len(scene)),
    "rows_daily": int(len(daily)),
    "rows_shortlist": int(len(shortlist)),
    "outputs": {"daily_csv": str(daily_csv), "shortlist_csv": str(short_csv), "plot_png": str(plot_path)},
}
manifest_path = Path(OUTPUT_DIR) / "clear_sky_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved:", daily_csv)
print("Saved:", short_csv)
print("Saved:", plot_path)
print("Saved:", manifest_path)
